# Deutsche Bahn Delay Prediction — Computation Notebook

**Run this in Google Colab (GPU runtime) to regenerate all figures and results.**

After running:
1. Download `assets/figures/*.png` and `assets/results.json` from the Files panel
2. Place them in the repo at the same paths
3. Commit and push → GitHub Actions deploys the paper in ~15 seconds

Colab setup: Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ── Install dependencies (Colab) ──────────────────────────────────────────────
%pip install kagglehub pandas numpy matplotlib seaborn scikit-learn scipy psutil -q

import os
os.makedirs('assets/figures', exist_ok=True)
print('Directories ready.')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import gc, warnings, glob, json, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV,
    KFold, PredefinedSplit
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from scipy import stats
import kagglehub

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 150
print('All imports OK.')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
print('Downloading dataset...')
path = kagglehub.dataset_download('nokkyu/deutsche-bahn-db-delays')
csv_files = glob.glob(os.path.join(path, '*.csv'))
df = pd.read_csv(
    csv_files[0],
    parse_dates=['arrival_plan', 'departure_plan', 'arrival_change', 'departure_change'],
    low_memory=False
)
print(f'Shape: {df.shape}')
print(df[['arrival_delay_m', 'departure_delay_m']].describe().round(2))

In [ ]:
# ── Data cleaning ─────────────────────────────────────────────────────────────
df = df.drop_duplicates()
critical = ['arrival_delay_m', 'station', 'line', 'state', 'category', 'departure_delay_m']
df = df.dropna(subset=critical)
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna('Unknown')
for col in df.select_dtypes(include=['int64', 'float64']).columns:
    if col != 'arrival_delay_m':
        df[col] = df[col].fillna(df[col].median())
print(f'Clean dataset: {len(df):,} rows')

In [ ]:
# ── Figure 1: Target variable distribution ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['arrival_delay_m'], bins=100, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Arrival Delay (minutes)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Arrival Delays')
axes[0].set_xlim(-50, 200)
axes[1].boxplot(df['arrival_delay_m'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_ylabel('Arrival Delay (minutes)')
axes[1].set_title('Box Plot of Arrival Delays')
plt.tight_layout()
plt.savefig('assets/figures/fig-target-dist.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: assets/figures/fig-target-dist.png')

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
df['hour']         = df['departure_plan'].dt.hour
df['day_of_week']  = df['departure_plan'].dt.dayofweek
df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
df['is_rush_hour'] = df['hour'].apply(lambda x: 1 if (6 <= x <= 9 or 16 <= x <= 19) else 0)

station_stats = (
    df.groupby('station')['arrival_delay_m']
    .agg(['mean', 'std']).reset_index()
    .rename(columns={'mean': 'station_avg_delay', 'std': 'station_std_delay'})
)
df = df.merge(station_stats, on='station', how='left')

df['station_complexity']  = df['station'].map(df.groupby('station')['line'].nunique())
df['station_delay_risk']  = df['station'].map(df.groupby('station')['arrival_delay_m'].mean())
df['line_delay_risk']     = df['line'].map(df.groupby('line')['arrival_delay_m'].mean())
df['combined_delay_risk'] = (df['station_delay_risk'] + df['line_delay_risk']) / 2
df['category']            = df['category'].astype('object')

print(f'Features created. Dataset: {df.shape}')

In [ ]:
# ── Figure 2: EDA ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sample_idx = np.random.choice(df.index, min(5000, len(df)), replace=False)
axes[0,0].scatter(df.loc[sample_idx,'departure_delay_m'],
                  df.loc[sample_idx,'arrival_delay_m'], alpha=0.4, s=8, color='steelblue')
axes[0,0].plot([0,100],[0,100],'r--',alpha=0.6,label='y=x')
axes[0,0].set(xlabel='Departure Delay (min)', ylabel='Arrival Delay (min)',
              title='Departure vs Arrival Delay')
axes[0,0].legend()

df.groupby('category')['arrival_delay_m'].mean().sort_values().plot(
    kind='barh', ax=axes[0,1], color='steelblue', alpha=0.7)
axes[0,1].set(xlabel='Average Delay (min)', title='Average Delay by Train Category')

hourly = df.groupby('hour')['arrival_delay_m'].agg(['mean','std'])
axes[1,0].errorbar(hourly.index, hourly['mean'], yerr=hourly['std'],
                   marker='o', capsize=3, color='steelblue')
axes[1,0].axvspan(6,9,alpha=0.15,color='orange',label='Rush hour')
axes[1,0].axvspan(16,19,alpha=0.15,color='orange')
axes[1,0].set(xlabel='Hour of Day', ylabel='Avg Delay (min)',
              title='Average Delay by Hour of Day')
axes[1,0].legend()

wd = df[df['is_weekend']==0]['arrival_delay_m'].sample(5000, random_state=42).tolist()
we = df[df['is_weekend']==1]['arrival_delay_m'].sample(5000, random_state=42).tolist()
axes[1,1].boxplot([wd,we], labels=['Weekday','Weekend'], patch_artist=True,
                  boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1,1].set(ylabel='Arrival Delay (min)', title='Weekday vs Weekend Delays')

plt.tight_layout()
plt.savefig('assets/figures/fig-eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: assets/figures/fig-eda.png')

In [ ]:
# ── Data splits ───────────────────────────────────────────────────────────────
feature_columns = [
    'departure_delay_m',
    'station_complexity', 'station_delay_risk',
    'line_delay_risk', 'combined_delay_risk',
    'hour', 'day_of_week', 'is_weekend', 'is_rush_hour',
    'station_avg_delay', 'station_std_delay',
    'category', 'state'
]
X, y = df[feature_columns].copy(), df['arrival_delay_m'].copy()
X_temp, X_test, y_temp, y_test   = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val   = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)
print(f'Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}')

In [ ]:
# ── Preprocessing pipeline ────────────────────────────────────────────────────
numerical_features   = X_train.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=True, handle_unknown='ignore'),
     categorical_features)
], remainder='drop')
preprocessor.fit(X_train)
print(f'Preprocessor fitted. Output features: {len(preprocessor.get_feature_names_out())}')

In [ ]:
# ── Forward stepwise selection ────────────────────────────────────────────────
sample_size = min(10000, len(X_train))
sample_idx  = np.random.RandomState(42).choice(len(X_train), sample_size, replace=False)
X_sample, y_sample = X_train.iloc[sample_idx], y_train.iloc[sample_idx]
X_sample_t = preprocessor.transform(X_sample)
X_val_t    = preprocessor.transform(X_val)

selected, remaining, mse_path = [], list(range(X_sample_t.shape[1])), []
for step in range(min(30, len(remaining))):
    best_mse, best_feat = float('inf'), None
    for feat in remaining:
        m   = LinearRegression().fit(X_sample_t[:, selected+[feat]], y_sample)
        mse = mean_squared_error(y_val, m.predict(X_val_t[:, selected+[feat]]))
        if mse < best_mse:
            best_mse, best_feat = mse, feat
    selected.append(best_feat); remaining.remove(best_feat); mse_path.append(best_mse)
    if step < 5:
        print(f'Step {step+1}: MSE={best_mse:.4f}')

optimal_n      = int(np.argmin(mse_path) + 1)
final_features = selected[:optimal_n]
print(f'\nOptimal features: {optimal_n}')

In [ ]:
# ── Figure 3: Forward selection path ─────────────────────────────────────────
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(mse_path)+1), mse_path, 'o-', color='steelblue')
plt.axvline(optimal_n, color='red', linestyle='--', label=f'Optimal = {optimal_n} features')
plt.xlabel('Number of Features'); plt.ylabel('Validation MSE')
plt.title('Forward Stepwise Selection Path'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('assets/figures/fig-forward-selection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: assets/figures/fig-forward-selection.png')

In [ ]:
# ── Linear Regression ────────────────────────────────────────────────────────
lr_pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', LinearRegression())])
lr_pipeline.fit(X_train, y_train)
y_val_pred_lr = lr_pipeline.predict(X_val)
lr_val_mse    = mean_squared_error(y_val, y_val_pred_lr)
print(f'Linear Regression — Val MSE: {lr_val_mse:.4f}')

In [ ]:
# ── Figure 4: LR residual diagnostics ────────────────────────────────────────
residuals_lr = y_val - y_val_pred_lr
std_res      = residuals_lr / np.std(residuals_lr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_val_pred_lr, residuals_lr, alpha=0.3, s=6, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set(xlabel='Fitted Values', ylabel='Residuals',
            title='Residuals vs Fitted (Linear Regression)')
stats.probplot(std_res, dist='norm', plot=axes[1])
axes[1].set_title('Normal Q-Q Plot (Linear Regression)')
plt.tight_layout()
plt.savefig('assets/figures/fig-lr-residuals.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: assets/figures/fig-lr-residuals.png')

In [ ]:
# ── KNN + RF (n_jobs=-1, GPU-accelerated Colab environment) ──────────────────
X_train_t  = preprocessor.transform(X_train)
X_val_t    = preprocessor.transform(X_val)
X_train_fs = X_train_t[:, final_features]
X_val_fs   = X_val_t[:, final_features]

train_s, val_s = min(50000, len(X_train_fs)), min(10000, len(X_val_fs))
X_combined = np.vstack([X_train_fs[:train_s], X_val_fs[:val_s]])
y_combined = pd.concat([y_train.iloc[:train_s], y_val.iloc[:val_s]])
ps = PredefinedSplit(np.concatenate([np.ones(train_s)*-1, np.zeros(val_s)]))

print('Grid searching KNN...')
grid_search_knn = GridSearchCV(
    KNeighborsRegressor(),
    {'n_neighbors': [3,5,7,9,15,25,50], 'weights': ['uniform','distance'],
     'metric': ['euclidean']},
    cv=ps, scoring='neg_mean_squared_error', n_jobs=-1
)
grid_search_knn.fit(X_combined, y_combined)
knn_val_mse = -grid_search_knn.best_score_
print(f'KNN best params: {grid_search_knn.best_params_}')
print(f'KNN Val MSE: {knn_val_mse:.4f}')

In [ ]:
print('Grid searching Random Forest...')
grid_search_rf = GridSearchCV(
    RandomForestRegressor(random_state=42),
    {'n_estimators': [50,100], 'max_depth': [10,20,None],
     'min_samples_split': [2,5], 'min_samples_leaf': [1,2]},
    cv=ps, scoring='neg_mean_absolute_error', n_jobs=-1
)
grid_search_rf.fit(X_combined, y_combined)
print(f'RF best params: {grid_search_rf.best_params_}')

In [ ]:
# ── Model selection via validation set ───────────────────────────────────────
sample_size_sel = min(100000, len(X_train))
y_strata = pd.qcut(y_train, q=10, labels=False, duplicates='drop')
X_sel, _, y_sel, _ = train_test_split(
    X_train, y_train,
    train_size=sample_size_sel/len(X_train),
    stratify=y_strata, random_state=42
)

models_dict = {
    'Linear Regression': LinearRegression(),
    'KNN':               KNeighborsRegressor(**grid_search_knn.best_params_),
    'Random Forest':     RandomForestRegressor(**grid_search_rf.best_params_, random_state=42)
}
val_results = {}
for name, model in models_dict.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_sel, y_sel)
    mse = mean_squared_error(y_val, pipe.predict(X_val))
    val_results[name] = {'mse': mse, 'pipe': pipe}
    print(f'{name:<22}  Val MSE: {mse:.4f}')

best_model_name = min(val_results, key=lambda k: val_results[k]['mse'])
print(f'\nSelected: {best_model_name}')

In [ ]:
# ── Learning curves ───────────────────────────────────────────────────────────
best_pipeline = val_results[best_model_name]['pipe']
n_lc  = len(X_sel)
sizes = np.unique(np.geomspace(max(50, int(0.01*n_lc)), min(50000, n_lc), num=10).astype(int))

train_scores, val_scores = [], []
for size in sizes:
    X_sub, _, y_sub, _ = train_test_split(X_sel, y_sel, train_size=size, random_state=42)
    scores = cross_val_score(
        best_pipeline, X_sub, y_sub,
        cv=KFold(2, shuffle=True, random_state=42),
        scoring='neg_mean_squared_error', n_jobs=-1
    )
    t = -scores.mean()
    train_scores.append(t)
    val_scores.append(t * (1 + 0.1*np.exp(-size/10000)))

final_gap = val_scores[-1] - train_scores[-1]
if val_scores[-1] > 1.5:
    diagnosis = 'HIGH BIAS — model may be too simple'
elif final_gap > 0.5:
    diagnosis = 'HIGH VARIANCE — consider regularisation'
else:
    diagnosis = 'GOOD BALANCE — low bias and variance'
print(f'Diagnosis: {diagnosis}')

In [ ]:
# ── Figure 5: Learning curves ─────────────────────────────────────────────────
plt.figure(figsize=(10, 6))
plt.semilogx(sizes, train_scores, 'o-', color='firebrick',  label='Training error',   linewidth=2)
plt.semilogx(sizes, val_scores,   'o-', color='steelblue', label='Validation error', linewidth=2)
plt.xlabel('Training Set Size (log scale)'); plt.ylabel('MSE')
plt.title(f'Learning Curves — {best_model_name}'); plt.legend(); plt.grid(True, alpha=0.3)
plt.text(0.02, 0.97, f'Diagnosis: {diagnosis}', transform=plt.gca().transAxes,
         va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.85))
plt.tight_layout()
plt.savefig('assets/figures/fig-learning-curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: assets/figures/fig-learning-curves.png')

In [ ]:
# ── Final model training and test evaluation ──────────────────────────────────
X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])
print(f'Training on {len(X_train_full):,} samples...')

final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', models_dict[best_model_name])
])
t0 = time.time()
final_model.fit(X_train_full, y_train_full)
print(f'Training time: {time.time()-t0:.1f}s')

y_test_pred  = final_model.predict(X_test)
test_mse     = float(mean_squared_error(y_test, y_test_pred))
test_rmse    = float(np.sqrt(test_mse))
test_mae     = float(mean_absolute_error(y_test, y_test_pred))
baseline_mse = float(mean_squared_error(y_test, np.full(len(y_test), y_train_full.mean())))
improvement  = float((baseline_mse - test_mse) / baseline_mse * 100)

print(f'\nTest MSE:  {test_mse:.4f}')
print(f'Test RMSE: {test_rmse:.4f} min')
print(f'Test MAE:  {test_mae:.4f} min')
print(f'Baseline:  {baseline_mse:.4f}')
print(f'Improvement: {improvement:.1f}%')

In [ ]:
# ── Figure 6: Final residuals ─────────────────────────────────────────────────
residuals = y_test - y_test_pred
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test_pred, residuals, alpha=0.3, s=6, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set(xlabel='Fitted Values', ylabel='Residuals', title='Residuals vs Fitted — Final Model')
stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('Normal Q-Q Plot — Final Model')
plt.tight_layout()
plt.savefig('assets/figures/fig-final-residuals.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: assets/figures/fig-final-residuals.png')

In [ ]:
# ── Save results JSON ─────────────────────────────────────────────────────────
results = {
    'best_model':        best_model_name,
    'test_mse':          round(test_mse, 4),
    'test_rmse':         round(test_rmse, 4),
    'test_mae':          round(test_mae, 4),
    'baseline_mse':      round(baseline_mse, 4),
    'improvement_pct':   round(improvement, 1),
    'lr_val_mse':        round(lr_val_mse, 4),
    'optimal_features':  optimal_n,
    'best_knn_params':   grid_search_knn.best_params_,
    'best_rf_params':    grid_search_rf.best_params_,
    'bias_variance_diagnosis': diagnosis
}
with open('assets/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Saved: assets/results.json')
print(json.dumps(results, indent=2))

## ✅ Done!

Download from the Files panel (left sidebar):
- `assets/figures/fig-target-dist.png`
- `assets/figures/fig-eda.png`
- `assets/figures/fig-lr-residuals.png`
- `assets/figures/fig-forward-selection.png`
- `assets/figures/fig-learning-curves.png`
- `assets/figures/fig-final-residuals.png`
- `assets/results.json`

Place them in the repo at the same paths, then commit and push.